# CS781 Capstone Phase 2 — named entities

Run this on Kaggle with a GPU.

1. New notebook, accelerator GPU T4, upload this file.
2. Add a secret named `HF` with your Hugging Face token. On huggingface.co, accept the terms for `Exploration-Lab/CS-781-Capstone` before the run.
3. Save Version, then Save & Run All.
4. Download `submission.zip` from Output and submit that zip. It contains `prediction.json`.

This phase trains its own model. It does not load the Phase 1 checkpoint, so it can be a separate Save Version.


In [ ]:
import os
import json
import re
import random
import zipfile
from collections import Counter

import torch
import torch.nn as nn
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
REPO = "Exploration-Lab/CS-781-Capstone"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = "/kaggle/working/cs781_phase2"
os.makedirs(OUT, exist_ok=True)
print("device", DEVICE)


def fetch(name):
    path = hf_hub_download(repo_id=REPO, filename=name, repo_type="dataset", token=HF_TOKEN)
    print("got", name)
    return path


def read_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        first = handle.read(1)
        handle.seek(0)
        if first in "[{":
            return json.load(handle)
        return [json.loads(line) for line in handle if line.strip()]


def as_mapping(payload):
    if isinstance(payload, dict):
        if any(key in payload for key in ("text", "entities", "annotations", "spans")):
            key = str(payload.get("id") or payload.get("doc_id") or "0")
            return {key: payload}
        return {str(key): value for key, value in payload.items()}
    mapping = {}
    for index, item in enumerate(payload):
        if isinstance(item, str):
            mapping[str(index)] = item
        elif isinstance(item, dict):
            key = str(item.get("id") or item.get("doc_id") or index)
            mapping[key] = item
    return mapping


def text_of(value):
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        for key in ("text", "sentence", "content", "judgment"):
            if isinstance(value.get(key), str):
                return value[key]
    return None


def spans_of(value):
    raw = value
    if isinstance(value, dict):
        raw = None
        for key in ("entities", "annotations", "spans", "labels", "gold"):
            if key in value:
                raw = value[key]
                break
        if raw is None:
            return []
    spans = []
    if not isinstance(raw, list):
        return spans
    for item in raw:
        if isinstance(item, (list, tuple)) and len(item) >= 3:
            spans.append((int(item[0]), int(item[1]), str(item[2])))
        elif isinstance(item, dict) and "start" in item and "end" in item:
            label = item.get("label") or item.get("type") or item.get("entity")
            spans.append((int(item["start"]), int(item["end"]), str(label)))
    return spans


def words_of(text):
    return [(match.start(), match.end()) for match in re.finditer(r"\S+", text)]


def tags_for(words, spans, label_set):
    tags = ["O"] * len(words)
    for start, end, label in spans:
        if label not in label_set:
            continue
        first = True
        for index, (left, right) in enumerate(words):
            if right <= start or left >= end:
                continue
            tags[index] = ("B-" if first else "I-") + label
            first = False
    return tags


def spans_from_tags(words, tags):
    found = []
    index = 0
    while index < len(words):
        tag = tags[index]
        if tag.startswith("B-") or tag.startswith("I-"):
            label = tag[2:]
            start = words[index][0]
            cursor = index + 1
            while cursor < len(words) and tags[cursor] == "I-" + label:
                cursor += 1
            end = words[cursor - 1][1]
            found.append([start, end, label])
            index = cursor
        else:
            index += 1
    return found


with open(fetch("Phase_2_LNER/Train/labels.txt"), "r", encoding="utf-8") as handle:
    LABELS = [line.strip() for line in handle if line.strip()]
print("labels", LABELS)
if len(LABELS) != 12:
    raise SystemExit("labels.txt must contain 12 labels")
LABEL_SET = set(LABELS)

documents = []
for fold in (1, 2):
    texts = as_mapping(read_json(fetch(f"Phase_2_LNER/Train/fold{fold}-text.json")))
    golds = as_mapping(read_json(fetch(f"Phase_2_LNER/Train/fold{fold}-gold.json")))
    print("fold", fold, "texts", len(texts), "golds", len(golds))
    for doc_id, payload in texts.items():
        text = text_of(payload)
        if text is None:
            continue
        spans = spans_of(golds.get(doc_id, []))
        words = words_of(text)
        tags = tags_for(words, spans, LABEL_SET)
        documents.append((doc_id, text, words, tags))

if not documents:
    raise SystemExit("no training documents parsed; check the printed fold sizes")

print("documents", len(documents), "example tags", documents[0][3][:12])

word_counts = Counter()
for _doc_id, text, words, _tags in documents:
    for left, right in words:
        word_counts[text[left:right]] += 1

itos = ["<pad>", "<unk>"] + [word for word, count in word_counts.most_common(30000) if count >= 2]
stoi = {word: index for index, word in enumerate(itos)}
tag_names = ["O"] + [prefix + label for label in LABELS for prefix in ("B-", "I-")]
tag_index = {name: index for index, name in enumerate(tag_names)}
print("vocab", len(itos), "tags", len(tag_names))


def encode_words(text, words):
    ids = []
    for left, right in words:
        ids.append(stoi.get(text[left:right], 1))
    return ids


class Tagger(nn.Module):
    def __init__(self, vocab, n_tags):
        super().__init__()
        self.embed = nn.Embedding(vocab, 128, padding_idx=0)
        self.lstm = nn.LSTM(128, 256, num_layers=2, bidirectional=True, batch_first=True, dropout=0.1)
        self.out = nn.Linear(512, n_tags)

    def forward(self, token_ids, lengths):
        x = self.embed(token_ids)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _state = self.lstm(packed)
        y, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        return self.out(y)


model = Tagger(len(itos), len(tag_names)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

examples = []
for doc_id, text, words, tags in documents:
    ids = encode_words(text, words)
    if not ids:
        continue
    target = [tag_index.get(tag, 0) for tag in tags]
    examples.append((doc_id, text, words, ids, target))

random.shuffle(examples)
cut = max(1, int(0.9 * len(examples)))
train_rows = examples[:cut]
dev_rows = examples[cut:]
print("train", len(train_rows), "dev", len(dev_rows))


def run_epoch(rows, train):
    model.train(train)
    total = 0.0
    seen = 0
    order = list(rows)
    if train:
        random.shuffle(order)
    batch = 16
    for start in range(0, len(order), batch):
        chunk = order[start : start + batch]
        width = max(len(item[3]) for item in chunk)
        token_ids = torch.zeros(len(chunk), width, dtype=torch.long, device=DEVICE)
        targets = torch.full((len(chunk), width), -100, dtype=torch.long, device=DEVICE)
        lengths = []
        for row, item in enumerate(chunk):
            ids = item[3][:400]
            gold = item[4][:400]
            token_ids[row, : len(ids)] = torch.tensor(ids, device=DEVICE)
            targets[row, : len(gold)] = torch.tensor(gold, device=DEVICE)
            lengths.append(len(ids))
        lengths = torch.tensor(lengths, dtype=torch.long)
        logits = model(token_ids, lengths)
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        total += float(loss) * len(chunk)
        seen += len(chunk)
    return total / max(1, seen)


for epoch in range(4):
    train_loss = run_epoch(train_rows, True)
    dev_loss = run_epoch(dev_rows, False)
    print(f"epoch {epoch + 1} train {train_loss:.4f} dev {dev_loss:.4f}")


def predict_doc(text):
    words = words_of(text)
    if not words:
        return []
    ids = encode_words(text, words)[:400]
    words = words[:400]
    token_ids = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    lengths = torch.tensor([len(ids)], dtype=torch.long)
    model.eval()
    with torch.no_grad():
        logits = model(token_ids, lengths)
    pred = logits[0, : len(ids)].argmax(dim=-1).tolist()
    tags = [tag_names[index] for index in pred]
    return spans_from_tags(words, tags)


matched = 0
gold_n = 0
pred_n = 0
for _doc_id, text, words, _ids, target in dev_rows[:80]:
    full_words = words_of(text)
    gold_spans = set(tuple(item) for item in spans_from_tags(full_words, [tag_names[i] for i in target]))
    pred_spans = set(tuple(item) for item in predict_doc(text))
    matched += len(gold_spans & pred_spans)
    gold_n += len(gold_spans)
    pred_n += len(pred_spans)
precision = matched / pred_n if pred_n else 0.0
recall = matched / gold_n if gold_n else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print(f"dev strict micro-F1 {f1:.4f}")

test_map = as_mapping(read_json(fetch("Phase_2_LNER/Test/fold3-text.json")))
predictions = {}
for index, (doc_id, payload) in enumerate(test_map.items()):
    text = text_of(payload) or ""
    predictions[str(doc_id)] = predict_doc(text)
    if index % 50 == 0:
        print("predicted", index, "of", len(test_map))

pred_path = os.path.join(OUT, "prediction.json")
zip_path = os.path.join(OUT, "submission.zip")
with open(pred_path, "w", encoding="utf-8") as handle:
    json.dump(predictions, handle, ensure_ascii=False)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pred_path, arcname="prediction.json")
print("zip", zip_path, "docs", len(predictions))
